# End to end transformer fine-tuning

## Dataset preparation

In [1]:
from datasets import load_dataset
dataset = load_dataset("Edoh/manim_python")

/home/lai/Documents/domain-specific-sml/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 599
    })
    test: Dataset({
        features: ['instruction', 'output'],
        num_rows: 51
    })
})

In [3]:
dataset['train'][0]

{'instruction': "Create a new scene named 'MyScene'.",
 'output': 'from manim import * class MyScene(Scene): def construct(self): pass'}

In [4]:
# Load tokenizer
from transformers import GPT2Tokenizer
model_name = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [5]:
def preprocess_data(example):
    inputs = [
        f'instruction": {instr}\n Output: {out}' for instr, out in zip(example["instruction"], example["output"])
    ]
    tokenized = tokenizer(inputs, truncation=True, max_length=512, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

## Tại sao cần tokenized["labels"] = tokenized["input_ids"].copy()?

  ### Bối cảnh: Đang fine-tune GPT-2

  Notebook này đang fine-tune model GPT-2 — một Causal Language Model (mô hình ngôn ngữ tự hồi quy). Nhiệm vụ huấn luyện của GPT-2 là: dự đoán token tiếp theo
  dựa trên các token trước đó.

  ### input_ids là gì?

  Khi bạn gọi tokenizer(inputs, ...), kết quả trả về là một dictionary chứa:

  • input_ids: danh sách các token ID (số nguyên) đại diện cho văn bản đầu vào.
  • attention_mask: mask cho biết token nào là thật, token nào là padding.

  ### labels là gì và tại sao lại bằng input_ids?

  Thư viện Hugging Face Transformers yêu cầu trường labels để tính loss (hàm mất mát) trong quá trình huấn luyện:

  1. Không có labels → model chỉ chạy forward pass, không tính loss, không học được gì.
  2. Có labels → model sẽ tính Cross-Entropy Loss giữa dự đoán và labels.
  3. labels = input_ids vì với Causal LM, mục tiêu huấn luyện là:
  │ Cho chuỗi token [A, B, C, D], model cần học:
  │
  │     • Từ A → dự đoán B
  │     • Từ A, B → dự đoán C
  │     • Từ A, B, C → dự đoán D
  Nói cách khác, đầu vào và đầu ra (target) là cùng một chuỗi, chỉ lệch nhau 1 vị trí. Hugging Face tự động xử lý việc dịch (shift) sang phải 1 vị trí bên trong
  model, nên bạn chỉ cần gán labels = input_ids.

  ### Tại sao dùng .copy()?

  Dùng .copy() để tạo một bản sao độc lập. Nếu không copy, labels và input_ids sẽ trỏ đến cùng một vùng nhớ — khi thay đổi một cái (ví dụ, mask padding tokens
  trong labels thành -100), cái còn lại cũng bị ảnh hưởng.

  ### Tóm tắt luồng hoạt động

    Input text: "instruction: Create a scene\n Output: from manim import ..."
                        ↓
                tokenizer(...)
                        ↓
            input_ids = [15, 42, 88, 103, ...]   ← token hóa văn bản
            labels    = [15, 42, 88, 103, ...]   ← bản sao, dùng làm target
                        ↓
                Trong model GPT-2:
                - Input:  [15, 42, 88, 103]  → Model dự đoán token tiếp theo
                - Target: [42, 88, 103, ...]  → (tự động shift bên trong)
                - Loss = CrossEntropy(dự đoán, target)

  │ Kết luận: labels chính là đáp án mà model cần học để dự đoán. Với Causal LM như GPT-2, đáp án chính là chuỗi token đầu vào (dịch phải 1 vị trí), nên labels =
  │ input_ids.copy().

In [6]:
tokenized_datasets = dataset.map(preprocess_data, batched=True, remove_columns=dataset["train"].column_names)

In [7]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 599
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 51
    })
})

In [8]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 599
    })
    test: Dataset({
        features: ['instruction', 'output'],
        num_rows: 51
    })
})

In [9]:
# Fine tuning
from transformers import GPT2LMHeadModel

def model_init():
    return GPT2LMHeadModel.from_pretrained(model_name, device_map='auto')

In [10]:
# Cấu hình các tham số huấn luyện (training arguments)
from transformers import TrainingArguments

training_args = TrainingArguments(

    # output_dir (str, bắt buộc): Thư mục lưu kết quả huấn luyện
    # - Nơi lưu checkpoint, model cuối cùng, log, v.v.
    # - Giá trị: đường dẫn bất kỳ, ví dụ: "./my-model", "/data/output"
    output_dir="./gpt2-manim-python-finetuned",

    # eval_strategy (str): Chiến lược đánh giá (evaluation) trên tập validation
    # - 'no'    : Không đánh giá trong quá trình huấn luyện
    # - 'steps' : Đánh giá sau mỗi N bước (cấu hình bởi eval_steps)
    # - 'epoch' : Đánh giá sau mỗi epoch (1 epoch = 1 lượt duyệt toàn bộ tập train)
    eval_strategy='epoch',

    # save_strategy (str): Chiến lược lưu checkpoint (bản sao model tại thời điểm huấn luyện)
    # - 'no'    : Không lưu checkpoint
    # - 'steps' : Lưu sau mỗi N bước (cấu hình bởi save_steps)
    # - 'epoch' : Lưu sau mỗi epoch
    # Lưu ý: save_strategy phải cùng loại với eval_strategy khi dùng load_best_model_at_end=True
    save_strategy='epoch',

    # logging_strategy (str): Chiến lược ghi log (loss, learning rate, v.v.)
    # - 'no'    : Không ghi log
    # - 'steps' : Ghi log sau mỗi N bước (cấu hình bởi logging_steps)
    # - 'epoch' : Ghi log sau mỗi epoch
    logging_strategy='steps',

    # logging_steps (int hoặc float): Số bước giữa mỗi lần ghi log
    # - Chỉ có tác dụng khi logging_strategy='steps'
    # - Giá trị int: số bước cụ thể, ví dụ: 10, 50, 100, 500
    # - Giá trị float (0.0-1.0): tỷ lệ so với tổng số bước, ví dụ: 0.1 = 10% tổng bước
    # - Mặc định: 500
    logging_steps=100,

    # save_total_limit (int): Số lượng checkpoint tối đa được giữ lại
    # - Khi vượt quá giới hạn, các checkpoint cũ nhất sẽ bị xóa tự động
    # - Giúp tiết kiệm dung lượng ổ đĩa
    # - Giá trị: số nguyên dương bất kỳ, ví dụ: 1, 2, 3, 5
    # - Mặc định: None (không giới hạn, giữ tất cả checkpoint)
    save_total_limit=2,

    # load_best_model_at_end (bool): Tải lại model tốt nhất khi huấn luyện xong
    # - True  : Sau khi train xong, tự động load checkpoint có metric tốt nhất
    # - False : Giữ nguyên model ở trạng thái cuối cùng (mặc định)
    # - Yêu cầu: eval_strategy và save_strategy phải cùng giá trị
    load_best_model_at_end=True,

    # metric_for_best_model (str): Tên metric dùng để chọn model tốt nhất
    # - Quyết định checkpoint nào là "tốt nhất" khi load_best_model_at_end=True
    # - Giá trị phổ biến: "eval_loss", "accuracy", "f1", "precision", "recall"
    # - Có thể dùng bất kỳ metric nào trả về từ hàm compute_metrics
    # - Mặc định: "loss" (nếu không dùng compute_metrics) hoặc metric đầu tiên từ compute_metrics
    metric_for_best_model="eval_loss",

    # greater_is_better (bool): Hướng so sánh metric — giá trị lớn hơn có tốt hơn không?
    # - True  : Metric cao hơn = tốt hơn (ví dụ: accuracy, f1)
    # - False : Metric thấp hơn = tốt hơn (ví dụ: loss, perplexity)
    # - Ở đây dùng eval_loss nên False là đúng (loss thấp = model tốt)
    greater_is_better=False,

    # fp16 (bool): Sử dụng huấn luyện với độ chính xác hỗn hợp 16-bit (mixed precision)
    # - True  : Dùng float16 thay vì float32, giúp tăng tốc và tiết kiệm bộ nhớ GPU
    # - False : Dùng float32 đầy đủ (mặc định)
    # - Yêu cầu GPU hỗ trợ (NVIDIA có Tensor Cores: Volta trở lên, ví dụ V100, T4, A100)
    # - Lưu ý: Không dùng đồng thời fp16=True và bf16=True
    # - Tham số liên quan: bf16=True (dùng bfloat16, hỗ trợ trên Ampere+, ví dụ A100, H100)
    fp16=True,

    # report_to (str hoặc List[str]): Công cụ theo dõi thí nghiệm (experiment tracker)
    # - 'none'        : Không gửi log đến công cụ nào
    # - 'wandb'       : Gửi log đến Weights & Biases
    # - 'tensorboard' : Gửi log đến TensorBoard
    # - 'mlflow'      : Gửi log đến MLflow
    # - 'comet_ml'    : Gửi log đến Comet ML
    # - 'azure_ml'    : Gửi log đến Azure ML
    # - 'all'         : Gửi đến tất cả các công cụ đã cài đặt
    # - Có thể truyền danh sách, ví dụ: ['wandb', 'tensorboard']
    # - Mặc định: 'all' (gửi đến tất cả integration đã cài)
    report_to="none"
)

In [11]:
train_val_split = tokenized_datasets["train"].train_test_split(test_size=0.1)
tokenized_datasets["train"] = train_val_split["train"]
tokenized_datasets["validation"] = train_val_split["test"]
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 539
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 51
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 60
    })
})

In [12]:
# We can now initialize the Trainer with the training arguments and the tokenized training and evaluation data:
from transformers import (
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)

data_collarator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [13]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collarator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3178.01it/s]


In [16]:
# define the search space for hyperparameter tuning as follows:
def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [2, 4, 8]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 6),
        "warmup_steps": trial.suggest_int("warmup_steps", 0, 500),
        "gradient_accumulation_steps": trial.suggest_categorical("gradient_accumulation_steps", [1, 2, 4])
    }

In [ ]:
best_run = trainer.hyperparameter_search(
    direction="minimize",
    backend="optuna",
    n_trials=10,
    hp_space=hp_space,
    compute_objective=lambda metrics: metrics["eval_loss"]
)

[I 2026-08-25 09:10:07,973] A new study created in memory with name: no-name-337e8695-0aff-496b-ada8-9a03243fa69d
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5246.25it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
/home/lai/Documents/domain-specific-sml/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,3.507755
2,No log,2.745098
3,3.403333,1.795214
4,3.403333,1.000875
5,3.403333,0.596151
6,1.087009,0.420243


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.73it/s]
/home/lai/Documents/domain-specific-sml/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.85it/s]
/home/lai/Documents/domain-specific-sml/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.43it/s]
/home/lai/Documents/domain-specific-sml/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 

Epoch,Training Loss,Validation Loss
1,No log,0.525483


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.52it/s]
/home/lai/Documents/domain-specific-sml/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# Setup the best value:
for key, value in best_run.hyperparameters.items():
    setattr(training_args, key, value)

trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets.get("validation"),
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

trainer.train()
trainer.save_model("./gpt2-manim-python-finetuned")
tokenizer.save_pretrained("./gpt2-manim-python-finetuned")